In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

# Add src to path to load Dataset helper
sys.path.append(os.path.abspath("src"))
from explainers_lib.datasets import Dataset

In [2]:
RESULTS_FILE = "experiments/titanic_results.json"

def load_titanic_reference():
    """Re-loads the dataset to calculate scores against."""
    url = "https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv"
    df = pd.read_csv(url)
    df = df.drop(['Name'], axis=1)
    # Replicate the split from experiment
    target = 'Survived'
    X = df.drop(target, axis=1)
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, random_state=42)
    
    # We need the full training set (X_test in this weird split context) for KNN
    # In the experiment X_test was the "dataset", so we use that as reference
    ds = Dataset(X_test, y_test.values, X_test.columns.tolist(), 
                 categorical_features=['Sex', 'Pclass'], 
                 continuous_features=['Age', 'Fare', 'Parents/Children Aboard', 'Siblings/Spouses Aboard'])
    return ds

In [3]:
def calculate_scores_standalone(ds, original_vector, cf_vectors, cf_targets):
    """
    Re-implementation of scoring logic to make notebook runnable 
    without 'utils.scores' source code.
    """
    # 1. Prepare Data
    # Transform raw data for distance calculations
    # Note: Using raw vectors (inverse transformed) usually makes more sense for Euclidean/Manhattan
    # But for simplicity, we calculate on the encoded vectors stored in the JSON
    
    X_ref = ds.data # The "training" data
    y_ref = np.array(ds.target)
    
    # 2. Fit KNN
    # Feasibility: Distance to 3 nearest neighbors
    knn_feas = NearestNeighbors(n_neighbors=3).fit(X_ref)
    
    # Discriminative: Class of 9 nearest neighbors
    knn_disc = NearestNeighbors(n_neighbors=9).fit(X_ref)
    
    scores = []
    
    for i, cf_vec in enumerate(cf_vectors):
        # A. Proximity (L1 Distance / Manhattan)
        prox = np.sum(np.abs(cf_vec - original_vector))
        
        # B. Feasibility (Avg distance to 3 closest points in data)
        dists, _ = knn_feas.kneighbors([cf_vec])
        feas = np.mean(dists[0])
        
        # C. Discriminative Power (% of 9 neighbors sharing target class)
        _, indices = knn_disc.kneighbors([cf_vec])
        neighbor_classes = y_ref[indices[0]]
        # Calculate ratio of neighbors that match the CF's target class
        disc = np.mean(neighbor_classes == cf_targets[i])
        
        scores.append({
            "Proximity": prox,
            "Feasibility": feas,
            "Discriminative": disc
        })
        
    return pd.DataFrame(scores)

In [4]:
if not os.path.exists(RESULTS_FILE):
    print(f"File not found: {RESULTS_FILE}")
else:
    with open(RESULTS_FILE, 'r') as f:
        raw_data = json.load(f)

    ds = load_titanic_reference()
    
    # Group by Original Instance
    grouped_data = {}
    for entry in raw_data:
        # Create a signature for the original instance
        orig_key = tuple(entry['original_data'])
        if orig_key not in grouped_data:
            grouped_data[orig_key] = {
                'original': np.array(entry['original_data']),
                'cfs': []
            }
        grouped_data[orig_key]['cfs'].append(entry)

    print(f"Loaded {len(raw_data)} CFs across {len(grouped_data)} original instances.")

Loaded 88 CFs across 10 original instances.


In [7]:
# Define ScoreBased selectors to highlight
SCORE_BASED_SELECTORS = {"Pareto", "IdealPoint", "BalancedPoint", "TOPSIS"}

fig = go.Figure()
steps = []

# Iterate through each original instance to create animation/slider steps
for i, (key, group) in enumerate(grouped_data.items()):
    original_vec = group['original']
    cfs_list = group['cfs']
    
    cf_vectors = np.array([cf['cf_data'] for cf in cfs_list])
    cf_targets = np.array([cf['target_class'] for cf in cfs_list])
    
    # Calculate Axes
    scores_df = calculate_scores_standalone(ds, original_vec, cf_vectors, cf_targets)
    
    # Prepare metadata for hover/color
    hover_texts = []
    marker_colors = []
    marker_symbols = []
    
    for idx, cf in enumerate(cfs_list):
        selectors = set(cf['selected_by'])
        score_selectors = selectors.intersection(SCORE_BASED_SELECTORS)
        
        # Logic for coloring/styling
        if "BalancedPoint" in selectors:
            col = "red"
            sym = "diamond"
            size = 12
        elif "IdealPoint" in selectors:
            col = "orange"
            sym = "diamond"
            size = 10
        elif "TOPSIS" in selectors:
            col = "purple"
            sym = "square"
            size = 10
        elif "Pareto" in selectors:
            col = "blue"
            sym = "circle"
            size = 8
        else:
            col = "gray"
            sym = "circle-open"
            size = 6
            
        hover_text = (
            f"Explainer: {cf['explainer']}<br>"
            f"Selected By: {', '.join(selectors)}<br>"
            f"Prox: {scores_df.iloc[idx]['Proximity']:.2f}<br>"
            f"Feas: {scores_df.iloc[idx]['Feasibility']:.2f}<br>"
            f"Disc: {scores_df.iloc[idx]['Discriminative']:.2f}"
        )
        
        hover_texts.append(hover_text)
        marker_colors.append(col)
        marker_symbols.append(sym)

    # Add trace (visible only if it's the first group)
    visible = (i == 0)
    
    scatter = go.Scatter3d(
        x=scores_df['Proximity'],
        y=scores_df['Feasibility'],
        z=scores_df['Discriminative'],
        mode='markers',
        marker=dict(
            size=8,
            color=marker_colors,
            symbol=marker_symbols,
            opacity=0.8,
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=hover_texts,
        hoverinfo='text',
        name=f"Instance {i}"
    )
    
    fig.add_trace(scatter)
    
    # Create slider step
    step = dict(
        method="update",
        args=[{"visible": [False] * len(grouped_data)},
              {"title": f"Original Instance {i+1} (Target: {group['cfs'][0]['original_class']})"}],
        label=str(i+1)
    )
    step["args"][0]["visible"][i] = True  # Toggle this trace on
    steps.append(step)

# Setup Layout
fig.update_layout(
    title="Scores visualization",
    height=800,
    scene=dict(
        xaxis_title='Proximity',
        yaxis_title='Feasibility',
        zaxis_title='Discriminative Power'
    ),
    sliders=[dict(
        active=0,
        currentvalue={"prefix": "Instance: "},
        pad={"t": 50},
        steps=steps
    )]
)

# Ensure only first is visible initially
for i in range(len(fig.data)):
    fig.data[i].visible = (i == 0)

fig.show()